# 02 — Analysis Report (Deliverable B)

Run from the project root context (`notebooks/` as cwd). Relative paths only.

Uses cleaned `data/processed/master_train.csv` from Deliverable A. Each task sits under its ID heading with a number/table/chart plus brief interpretation.

At the end, findings are written to `reports/B_analysis_report.md`.


## Setup & load master table


In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

from src.cleaning import load_raw_events, prepare_events

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

master = pd.read_csv(ROOT / "data" / "processed" / "master_train.csv", parse_dates=["pickup_hour_eat"])
assert master["pickup_hour_eat"].dt.tz is not None

# Zone types from weekday hourly shape (B1.2) — used again in B2/B3
ZONE_TYPE = {
    "Merkato": "market",
    "Megenagna": "transport_hub",
    "Bole": "nightlife_airport",
    "Kazanchis": "business",
    "Arat Kilo": "business",
    "Lideta": "business",
    "Piassa": "business",
    "CMC": "residential",
    "Gerji": "residential",
    "Kolfe": "residential",
    "Sarbet": "residential",
    "Ayat": "residential",
}
master["zone_type"] = master["zone"].map(ZONE_TYPE)
master["day"] = master["pickup_hour_eat"].dt.floor("D")

findings: dict[str, str] = {}
print(master.shape)
print(master["pickup_hour_eat"].min(), "→", master["pickup_hour_eat"].max())
master.head()


(84614, 31)
2025-01-01 00:00:00+03:00 → 2025-10-31 23:00:00+03:00


,zone,pickup_hour_eat,trips,avg_fare_birr,avg_wait_min,active_drivers,record_id,data_type,temp_c,rain_mm,...,dow,is_weekend,day_of_month,is_payday_window,trend_day,rain_last_3h,rain_class,trips_lag_168,zone_type,day
0,Arat Kilo,2025-01-01 00:00:00+03:00,5.0,240.4,1.9,5.0,R-070502,observed,9.8,0.0,...,2,0,1,1,0,NaN,0,3.0,business,2025-01-01 00:00:00+03:00
1,Arat Kilo,2025-01-01 01:00:00+03:00,2.0,224.4,1.2,3.0,R-045764,observed,8.0,0.0,...,2,0,1,1,0,NaN,0,3.0,business,2025-01-01 00:00:00+03:00
2,Arat Kilo,2025-01-01 02:00:00+03:00,2.0,226.5,1.2,3.0,R-023866,observed,7.2,0.0,...,2,0,1,1,0,0.0,0,2.5,business,2025-01-01 00:00:00+03:00
3,Arat Kilo,2025-01-01 03:00:00+03:00,4.0,233.1,1.3,5.0,R-024252,NaN,7.2,0.0,...,2,0,1,1,0,0.0,0,3.0,business,2025-01-01 00:00:00+03:00
4,Arat Kilo,2025-01-01 04:00:00+03:00,5.0,244.2,1.2,6.0,R-005550,observed,6.7,0.0,...,2,0,1,1,0,0.0,0,3.0,business,2025-01-01 00:00:00+03:00


## B1 — Demand patterns


### B1.1 Volume by zone


In [2]:
b11 = (
    master.groupby("zone", as_index=False)
    .agg(
        total_trips=("trips", "sum"),
        mean_trips_per_zone_hour=("trips", "mean"),
        n_hours=("trips", "count"),
        first_hour=("pickup_hour_eat", "min"),
        last_hour=("pickup_hour_eat", "max"),
    )
    .sort_values("total_trips", ascending=False)
)
b11["share_pct"] = 100 * b11["total_trips"] / b11["total_trips"].sum()
b11


,zone,total_trips,mean_trips_per_zone_hour,n_hours,first_hour,last_hour,share_pct
9,Merkato,295018.5,41.043197,7188,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,12.128523
2,Bole,284047.5,39.472971,7196,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,11.677494
8,Megenagna,279665.0,38.863952,7196,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,11.497324
5,Kazanchis,230659.0,32.062691,7194,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,9.482636
7,Lideta,205029.0,28.468342,7202,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,8.428959
10,Piassa,203914.0,28.325323,7199,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,8.383121
3,CMC,188111.0,26.141051,7196,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,7.733442
4,Gerji,175867.0,24.429365,7199,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,7.230079
6,Kolfe,166349.0,23.116870,7196,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,6.838784
11,Sarbet,156989.5,21.837460,7189,2025-01-01 00:00:00+03:00,2025-10-31 23:00:00+03:00,6.454005


In [3]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(b11["zone"], b11["share_pct"], color="#1f4e79")
ax.set_xlabel("Share of city trips (%)")
ax.set_title("B1.1 — Zone demand share")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

ayat_note = (
    "Ayat launches mid-March (not full-period), so its share understates eventual weight."
    if b11.loc[b11["zone"] == "Ayat", "first_hour"].iloc[0] > master["pickup_hour_eat"].min()
    else ""
)
top3 = ", ".join(f"{r.zone} ({r.share_pct:.1f}%)" for r in b11.head(3).itertuples())
findings["B1.1"] = (
    f"Top zones by volume: {top3}. City mean trips/zone-hour ranges "
    f"{b11['mean_trips_per_zone_hour'].min():.1f}–{b11['mean_trips_per_zone_hour'].max():.1f}. {ayat_note}"
)
print(findings["B1.1"])


Top zones by volume: Merkato (12.1%), Bole (11.7%), Megenagna (11.5%). City mean trips/zone-hour ranges 18.0–41.0. Ayat launches mid-March (not full-period), so its share understates eventual weight.


### B1.2 Hour-of-day profile by zone type


In [4]:
wd = master[master["is_weekend"] == 0]
prof = wd.groupby(["zone", "hour"])["trips"].mean().unstack("hour")
prof_type = wd.groupby(["zone_type", "hour"])["trips"].mean().unstack("hour")

peak = prof_type.idxmax(axis=1).rename("peak_hour")
quiet = prof_type.idxmin(axis=1).rename("quiet_hour")
b12 = pd.concat([peak, quiet, prof_type.max(axis=1).rename("peak_mean"), prof_type.min(axis=1).rename("quiet_mean")], axis=1)
b12


,peak_hour,quiet_hour,peak_mean,quiet_mean
zone_type,,,,
business,18,2,77.513970,4.382867
market,13,0,91.356481,4.089862
nightlife_airport,6,2,58.273148,10.651869
residential,7,2,47.958415,3.734834
transport_hub,18,1,90.183721,7.817130


In [5]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for zt, row in prof_type.iterrows():
    ax.plot(row.index, row.values, marker="o", ms=3, label=zt)
ax.set_xlabel("Hour of day (EAT)")
ax.set_ylabel("Mean weekday trips")
ax.set_title("B1.2 — Weekday hourly profile by zone type")
ax.legend(fontsize=8, ncol=2)
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

lines = [
    f"{zt}: peaks {int(r.peak_hour):02d}:00 (mean {r.peak_mean:.1f}), quietest {int(r.quiet_hour):02d}:00"
    for zt, r in b12.iterrows()
]
findings["B1.2"] = (
    "Zone types from weekday shapes — residential (morning ~07:00), business (evening ~18:00), "
    "market (midday ~13:00), transport hub (evening), nightlife/airport (early morning + weekend lift). "
    + " | ".join(lines)
)
print(findings["B1.2"])


Zone types from weekday shapes — residential (morning ~07:00), business (evening ~18:00), market (midday ~13:00), transport hub (evening), nightlife/airport (early morning + weekend lift). business: peaks 18:00 (mean 77.5), quietest 02:00 | market: peaks 13:00 (mean 91.4), quietest 00:00 | nightlife_airport: peaks 06:00 (mean 58.3), quietest 02:00 | residential: peaks 07:00 (mean 48.0), quietest 02:00 | transport_hub: peaks 18:00 (mean 90.2), quietest 01:00


### B1.3 Weekday vs weekend


In [6]:
ww = master.groupby(["zone", "is_weekend"])["trips"].mean().unstack("is_weekend")
ww = ww.rename(columns={0: "weekday_mean", 1: "weekend_mean"})
ww["weekend_to_weekday"] = ww["weekend_mean"] / ww["weekday_mean"]
ww = ww.sort_values("weekend_to_weekday", ascending=False)
ww.round(3)


is_weekend,weekday_mean,weekend_mean,weekend_to_weekday
zone,,,
Bole,36.229,47.640,1.315
Gerji,23.879,25.812,1.081
Ayat,17.632,19.002,1.078
Kolfe,22.621,24.366,1.077
Sarbet,21.408,22.920,1.071
CMC,25.645,27.390,1.068
Merkato,42.504,37.359,0.879
Lideta,30.092,24.383,0.810
Megenagna,41.118,33.194,0.807


In [7]:
standout = ww["weekend_to_weekday"].idxmin()  # strongest weekend collapse
dow_shape = (
    master[master["zone"] == standout]
    .groupby("dow")["trips"]
    .mean()
    .reindex(range(7))
)
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].barh(ww.index, ww["weekend_to_weekday"], color="#2e7d32")
axes[0].axvline(1.0, color="gray", ls="--", lw=1)
axes[0].set_xlabel("Weekend / weekday mean trips")
axes[0].set_title("B1.3 — Weekend-to-weekday ratio")
axes[0].invert_yaxis()
axes[1].bar(dow_names, dow_shape.values, color="#1f4e79")
axes[1].set_title(f"Day-of-week shape — {standout}")
axes[1].set_ylabel("Mean trips")
plt.tight_layout()
plt.show()

busy = ", ".join(f"{z} ({r.weekend_to_weekday:.2f})" for z, r in ww.head(3).iterrows())
quiet_w = ", ".join(f"{z} ({r.weekend_to_weekday:.2f})" for z, r in ww.tail(3).iterrows())
findings["B1.3"] = (
    f"Weekend-busier: {busy}. Weekend-collapse: {quiet_w}. "
    f"{standout} stands out — weekday office/commerce demand drops sharply Sat–Sun "
    f"(means: " + ", ".join(f"{n}={v:.1f}" for n, v in zip(dow_names, dow_shape)) + ")."
)
print(findings["B1.3"])


Weekend-busier: Bole (1.31), Gerji (1.08), Ayat (1.08). Weekend-collapse: Kazanchis (0.51), Piassa (0.49), Arat Kilo (0.49). Arat Kilo stands out — weekday office/commerce demand drops sharply Sat–Sun (means: Mon=24.2, Tue=24.6, Wed=23.9, Thu=22.9, Fri=25.1, Sat=11.9, Sun=11.5).


### B1.4 Trend (January → October)


In [8]:
city_daily = master.groupby("day", as_index=False)["trips"].sum()
city_daily["week"] = city_daily["day"].dt.to_period("W").apply(lambda p: p.start_time)
weekly = city_daily.groupby("week", as_index=False)["trips"].sum().rename(columns={"trips": "weekly_trips"})
# drop partial first/last weeks for growth calc
full = weekly.iloc[1:-1].copy()
early = full.head(4)["weekly_trips"].mean()
late = full.tail(4)["weekly_trips"].mean()
growth_pct = 100 * (late / early - 1)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(weekly["week"], weekly["weekly_trips"], marker="o", ms=3, color="#1f4e79")
ax.set_title("B1.4 — City-wide weekly trips")
ax.set_ylabel("Trips / week")
plt.tight_layout()
plt.show()

weekly[["week", "weekly_trips"]].tail(8)


,week,weekly_trips
36,2025-09-08,65636.0
37,2025-09-15,63596.5
38,2025-09-22,69207.0
39,2025-09-29,67514.5
40,2025-10-06,65198.5
41,2025-10-13,65380.5
42,2025-10-20,65335.0
43,2025-10-27,52173.0


In [9]:
findings["B1.4"] = (
    f"Weekly city trips rise from ~{early:,.0f} (early full weeks) to ~{late:,.0f} "
    f"(late full weeks): +{growth_pct:.1f}%. "
    f"For November forecasts, a positive trend term (or recent lag levels) is essential — "
    f"a January baseline would under-predict by roughly that growth gap."
)
print(findings["B1.4"])
print(weekly.head(3).to_string(index=False))
print("...")
print(weekly.tail(3).to_string(index=False))


Weekly city trips rise from ~44,816 (early full weeks) to ~65,857 (late full weeks): +46.9%. For November forecasts, a positive trend term (or recent lag levels) is essential — a January baseline would under-predict by roughly that growth gap.
      week  weekly_trips
2024-12-30       31805.0
2025-01-06       42498.5
2025-01-13       44050.0
...
      week  weekly_trips
2025-10-13       65380.5
2025-10-20       65335.0
2025-10-27       52173.0


## B2 — Weather


### B2.1 Timezone check


In [10]:
city_wx = master.drop_duplicates("pickup_hour_eat")[["pickup_hour_eat", "temp_c", "rain_mm"]].copy()
city_wx["hour"] = city_wx["pickup_hour_eat"].dt.hour
temp_by_hour = city_wx.groupby("hour")["temp_c"].mean()
peak_temp_hour = int(temp_by_hour.idxmax())

# Rain–demand relationship under artificial clock shifts of weather (0–5h)
base = master[["pickup_hour_eat", "zone", "trips"]].copy()
wx = city_wx[["pickup_hour_eat", "rain_mm"]].copy()
shift_rows = []
for shift in range(0, 6):
    w = wx.copy()
    w["pickup_hour_eat"] = w["pickup_hour_eat"] + pd.Timedelta(hours=shift)
    tmp = base.merge(w, on="pickup_hour_eat", how="left")
    corr = tmp[["trips", "rain_mm"]].corr().iloc[0, 1]
    rainy = tmp.loc[tmp["rain_mm"] >= 0.5, "trips"].mean()
    dry = tmp.loc[tmp["rain_mm"] < 0.1, "trips"].mean()
    shift_rows.append(
        {
            "weather_shift_hours": shift,
            "corr_trips_rain": corr,
            "rainy_over_dry_mean": rainy / dry if dry else np.nan,
        }
    )
b21 = pd.DataFrame(shift_rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(temp_by_hour.index, temp_by_hour.values, marker="o", color="#c0392b")
axes[0].axvline(peak_temp_hour, color="gray", ls="--")
axes[0].set_title(f"Mean temp by hour (peak={peak_temp_hour}:00 EAT)")
axes[0].set_xlabel("Hour EAT")
axes[0].set_ylabel("°C")
axes[1].plot(b21["weather_shift_hours"], b21["corr_trips_rain"], marker="o", label="corr")
axes[1].plot(b21["weather_shift_hours"], b21["rainy_over_dry_mean"] - 1, marker="s", label="rainy/dry − 1")
axes[1].set_xlabel("Hours weather shifted later")
axes[1].set_title("B2.1 — Misalignment destroys rain signal")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
b21.round(3)


,weather_shift_hours,corr_trips_rain,rainy_over_dry_mean
0,0,0.174,1.781
1,1,0.105,1.557
2,2,0.086,1.454
3,3,0.053,1.274
4,4,0.021,1.097
5,5,-0.015,0.920


In [11]:
best = b21.iloc[0]
worst = b21.iloc[-1]
findings["B2.1"] = (
    f"Temperature peaks at {peak_temp_hour}:00 EAT — consistent with local afternoon heat, "
    f"so weather is correctly on Africa/Addis_Ababa after UTC→EAT conversion. "
    f"At 0h shift, corr(trips, rain)={best.corr_trips_rain:.3f} and rainy/dry={best.rainy_over_dry_mean:.2f}; "
    f"at +5h these fall to {worst.corr_trips_rain:.3f} and {worst.rainy_over_dry_mean:.2f}. "
    f"A 3h UTC mistake would materially weaken the rain feature."
)
print(findings["B2.1"])


Temperature peaks at 15:00 EAT — consistent with local afternoon heat, so weather is correctly on Africa/Addis_Ababa after UTC→EAT conversion. At 0h shift, corr(trips, rain)=0.174 and rainy/dry=1.78; at +5h these fall to -0.015 and 0.92. A 3h UTC mistake would materially weaken the rain feature.


### B2.2 Rain effect by zone type


In [12]:
dry_base = (
    master.loc[master["rain_mm"] <= 0]
    .groupby(["zone", "dow", "hour"])["trips"]
    .mean()
    .rename("dry_mean")
)
rainy = master.loc[master["rain_mm"] >= 0.5].merge(dry_base, on=["zone", "dow", "hour"], how="left")
rainy = rainy.dropna(subset=["dry_mean"])
rainy = rainy[rainy["dry_mean"] > 0]
rainy["rain_over_dry"] = rainy["trips"] / rainy["dry_mean"]

b22 = (
    rainy.groupby("zone_type")["rain_over_dry"]
    .agg(mean_ratio="mean", median_ratio="median", n_hours="count")
    .sort_values("mean_ratio", ascending=False)
)
b22.round(3)


,mean_ratio,median_ratio,n_hours
zone_type,,,
transport_hub,1.367,1.319,485
nightlife_airport,1.338,1.299,483
residential,1.333,1.282,2392
business,1.328,1.276,1945
market,0.828,0.814,484


In [13]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(b22.index, b22["mean_ratio"], color="#1565c0")
ax.axhline(1.0, color="gray", ls="--")
ax.set_ylabel("Mean trips / matched dry baseline")
ax.set_title("B2.2 — Rain uplift by zone type (≥0.5 mm vs dry same zone·dow·hour)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

up = b22[b22["mean_ratio"] >= 1]
down = b22[b22["mean_ratio"] < 1]
findings["B2.2"] = (
    "Matched rainy (≥0.5 mm) vs dry same zone·weekday·hour: "
    + (
        "lift in " + ", ".join(f"{zt} ×{r.mean_ratio:.2f}" for zt, r in up.iterrows())
        if len(up)
        else "no zone type lifts"
    )
    + (
        "; demand falls in " + ", ".join(f"{zt} ×{r.mean_ratio:.2f}" for zt, r in down.iterrows())
        if len(down)
        else ""
    )
    + ". Rain does not increase demand everywhere — Merkato (market) softens, consistent with outdoor trading pausing in wet weather."
)
print(findings["B2.2"])


Matched rainy (≥0.5 mm) vs dry same zone·weekday·hour: lift in transport_hub ×1.37, nightlife_airport ×1.34, residential ×1.33, business ×1.33; demand falls in market ×0.83. Rain does not increase demand everywhere — Merkato (market) softens, consistent with outdoor trading pausing in wet weather.


### B2.3 Rain dose-response


In [14]:
RAIN_LABEL = {0: "none", 1: "light", 2: "moderate", 3: "heavy"}
dose = master.merge(dry_base, on=["zone", "dow", "hour"], how="left")
dose = dose.dropna(subset=["dry_mean"])
dose = dose[dose["dry_mean"] > 0]
dose["ratio"] = dose["trips"] / dose["dry_mean"]
dose["rain_label"] = dose["rain_class"].map(RAIN_LABEL)

b23 = dose.groupby(["zone_type", "rain_label"])["ratio"].mean().unstack("rain_label")
# column order
b23 = b23.reindex(columns=["none", "light", "moderate", "heavy"])
b23.round(3)


rain_label,none,light,moderate,heavy
zone_type,,,,
business,1.0,1.155,1.284,1.548
market,1.0,0.974,0.856,0.682
nightlife_airport,1.0,1.102,1.310,1.542
residential,1.0,1.114,1.292,1.543
transport_hub,1.0,1.125,1.365,1.507


In [15]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(b23.columns))
width = 0.15
for i, (zt, row) in enumerate(b23.iterrows()):
    ax.plot(b23.columns, row.values, marker="o", label=zt)
ax.set_ylabel("Demand ratio vs dry baseline")
ax.set_title("B2.3 — Rain dose-response by zone type")
ax.legend(fontsize=8, ncol=2)
ax.axhline(1.0, color="gray", ls="--", lw=1)
plt.tight_layout()
plt.show()

# saturation note
city_dose = dose.groupby("rain_label")["ratio"].mean().reindex(["none", "light", "moderate", "heavy"])
step = city_dose.diff()
findings["B2.3"] = (
    f"City-wide dose-response (vs dry baseline): "
    + " → ".join(f"{k}={city_dose[k]:.2f}" for k in city_dose.index)
    + f". Light rain already lifts demand (+{100*(city_dose['light']-1):.0f}%); "
    + f"moderate and heavy keep rising (heavy {city_dose['heavy']:.2f}). "
    + "Not a pure straight line — early step from none→light is large relative to rainfall amount; "
    + "zone types differ in slope (see table)."
)
print(findings["B2.3"])
print(city_dose.round(3).to_string())


City-wide dose-response (vs dry baseline): none=1.00 → light=1.12 → moderate=1.26 → heavy=1.47. Light rain already lifts demand (+12%); moderate and heavy keep rising (heavy 1.47). Not a pure straight line — early step from none→light is large relative to rainfall amount; zone types differ in slope (see table).
rain_label
none        1.000
light       1.116
moderate    1.261
heavy       1.470


## B3 — Events & calendar


### B3.1 Public holidays


In [16]:
from src.cleaning import DEFAULT_DURATION_HOURS, EVENT_WINDOW

events = prepare_events(load_raw_events())
confirmed = events[events["status_clean"].str.contains("confirm", na=False)].copy()
holidays = confirmed[confirmed["event_type"] == "public_holiday"].drop_duplicates("event_id")

# Some raw holiday end timestamps span months (bad data). Resolve a single analysis day:
# - short windows: start day
# - known overrides for corrupted rows
# - Labour-Day-like swap (start ~23:59, end midnight months later): prefer end calendar day
HOLIDAY_DAY_OVERRIDE = {
    "EVT-0055": "2025-05-01",  # International Labour Day (start was 2025-01-05 23:59)
    "EVT-0001": "2025-01-07",  # Genna
    "EVT-0121": "2025-09-11",  # Enkutatash
}

def holiday_analysis_day(ev) -> pd.Timestamp:
    if ev["event_id"] in HOLIDAY_DAY_OVERRIDE:
        d = pd.Timestamp(HOLIDAY_DAY_OVERRIDE[ev["event_id"]], tz="Africa/Addis_Ababa")
        return d
    start = pd.Timestamp(ev["start_eat"]).tz_convert("Africa/Addis_Ababa")
    end = pd.Timestamp(ev["end_eat"]).tz_convert("Africa/Addis_Ababa")
    dur_h = (end - start).total_seconds() / 3600
    if dur_h <= 36:
        return start.normalize()
    if start.hour >= 22 and end.hour == 0 and end.minute == 0:
        return end.normalize()
    return start.normalize()

city_by_day = master.groupby("day")["trips"].sum()
holiday_days = {holiday_analysis_day(r) for _, r in holidays.iterrows()}

rows = []
for _, ev in holidays.iterrows():
    d = holiday_analysis_day(ev)
    day_trips = float(city_by_day.get(d, np.nan))
    same = city_by_day[
        (city_by_day.index.dayofweek == d.dayofweek) & (~city_by_day.index.isin(holiday_days))
    ]
    near = same[np.abs((same.index - d).days) <= 28]
    ref = float(near.mean()) if len(near) else np.nan
    day_z = master.loc[master["day"] == d].groupby("zone")["trips"].sum()
    ref_z = (
        master.loc[master["day"].isin(near.index)]
        .groupby(["zone", "day"])["trips"]
        .sum()
        .groupby("zone")
        .mean()
    )
    local = (day_z / ref_z).replace([np.inf, -np.inf], np.nan).dropna().sort_values()
    rows.append(
        {
            "event_id": ev["event_id"],
            "event_name": ev["event_name"],
            "date": d.date(),
            "city_trips": day_trips,
            "near_same_weekday_mean": ref,
            "ratio_vs_near": day_trips / ref if ref else np.nan,
            "strongest_local_drop": f"{local.index[0]} ({local.iloc[0]:.2f})" if len(local) else "",
            "strongest_local_rise": f"{local.index[-1]} ({local.iloc[-1]:.2f})" if len(local) else "",
        }
    )
b31 = pd.DataFrame(rows).dropna(subset=["ratio_vs_near"]).sort_values("ratio_vs_near")
b31


,event_id,event_name,date,city_trips,near_same_weekday_mean,ratio_vs_near,strongest_local_drop,strongest_local_rise
7,EVT-0001,Genna (Ethiopian Christmas),2025-01-07,4923.0,6822.125000,0.721623,Piassa (0.40),Bole (1.12)
0,EVT-0045,Good Friday,2025-04-18,6421.5,8250.625000,0.778305,Kazanchis (0.44),Bole (1.28)
3,EVT-0079,Eid al-Adha,2025-06-06,7262.5,8853.062500,0.820338,Piassa (0.44),Bole (1.14)
1,EVT-0040,Eid al-Fitr,2025-03-31,6434.0,7702.875000,0.835273,Piassa (0.46),Bole (1.33)
10,EVT-0121,Enkutatash (Ethiopian New Year),2025-09-11,7891.0,9406.285714,0.838907,Merkato (0.45),Bole (1.50)
8,EVT-0115,Mawlid,2025-09-04,7927.0,9366.357143,0.846327,Arat Kilo (0.34),Bole (1.56)
9,EVT-0078,Downfall of the Derg,2025-05-28,7059.5,8299.937500,0.850549,Kazanchis (0.43),Kolfe (1.37)
2,EVT-0055,International Labour Day,2025-05-01,6767.0,7934.625000,0.852844,Arat Kilo (0.44),Bole (1.31)
5,EVT-0061,Patriots' Victory Day,2025-05-05,6377.0,7284.812500,0.875383,Kazanchis (0.49),Ayat (1.33)
6,EVT-0048,Fasika (Ethiopian Easter),2025-04-20,6280.0,6444.687500,0.974446,Gerji (0.82),Bole (1.09)


In [17]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(b31["event_name"].astype(str), b31["ratio_vs_near"], color="#6a1b9a")
ax.axvline(1.0, color="gray", ls="--")
ax.set_xlabel("City trips / nearby same-weekday mean")
ax.set_title("B3.1 — Public holiday impact (ranked)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

reducers = b31[b31["ratio_vs_near"] < 0.98]
lifters = b31[b31["ratio_vs_near"] >= 1.02]
findings["B3.1"] = (
    f"Holiday city-day vs nearby same weekdays — rank (low→high): "
    + "; ".join(f"{r.event_name} ({r.ratio_vs_near:.2f})" for r in b31.itertuples())
    + ". "
    + (
        f"{len(reducers)}/{len(b31)} holidays clearly reduce demand; "
        if len(reducers)
        else "No clear city-wide drops; "
    )
    + (
        f"not all do — e.g. {', '.join(lifters['event_name'].astype(str))} hold flat/up. "
        if len(lifters)
        else "nearly all reduce demand. "
    )
    + "Local effects vary (see strongest_local_drop/rise): business zones often drop more than residential."
)
print(findings["B3.1"])


Holiday city-day vs nearby same weekdays — rank (low→high): Genna (Ethiopian Christmas) (0.72); Good Friday (0.78); Eid al-Adha (0.82); Eid al-Fitr (0.84); Enkutatash (Ethiopian New Year) (0.84); Mawlid (0.85); Downfall of the Derg (0.85); International Labour Day (0.85); Patriots' Victory Day (0.88); Fasika (Ethiopian Easter) (0.97); Meskel (Finding of the True Cross) (0.99); Timkat (Epiphany) (1.00); Adwa Victory Day (1.02). 10/13 holidays clearly reduce demand; not all do — e.g. Adwa Victory Day hold flat/up. Local effects vary (see strongest_local_drop/rise): business zones often drop more than residential.


### B3.2 Event-window study (football)


In [18]:
# Vectorized baselines: mean trips by zone·dow·hour outside football windows
fb_base = (
    master.loc[master["is_football_window"] == 0]
    .groupby(["zone", "dow", "hour"])["trips"]
    .mean()
    .rename("base_mean")
)

fb_events = confirmed[confirmed["event_type"] == "football_match"].drop_duplicates("event_id")
# one row per event-zone (already exploded)
fb_ez = confirmed[confirmed["event_type"] == "football_match"].dropna(subset=["zone"])

pieces = []
for _, ev in fb_ez.iterrows():
    z, start, end = ev["zone"], ev["start_eat"], ev["end_eat"]
    windows = {
        "pre_2h": (start - pd.Timedelta(hours=2), start),
        "during": (start, end),
        "post_2h": (end, end + pd.Timedelta(hours=2)),
    }
    for label, (t0, t1) in windows.items():
        sub = master[(master["zone"] == z) & (master["pickup_hour_eat"] >= t0) & (master["pickup_hour_eat"] < t1)]
        if sub.empty:
            continue
        tmp = sub.merge(fb_base, on=["zone", "dow", "hour"], how="left")
        tmp = tmp[tmp["base_mean"] > 0]
        if tmp.empty:
            continue
        pieces.append(pd.DataFrame({"window": label, "ratio": tmp["trips"] / tmp["base_mean"]}))

fb_df = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=["window", "ratio"])
b32 = fb_df.groupby("window")["ratio"].agg(mean="mean", median="median", n="count")
b32 = b32.reindex(["pre_2h", "during", "post_2h"])
b32


,mean,median,n
window,,,
pre_2h,2.165571,2.098810,82
during,1.441110,1.291513,83
post_2h,2.854945,2.180880,84


In [19]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(b32.index.astype(str), b32["mean"], color="#e65100")
ax.axhline(1.0, color="gray", ls="--")
ax.set_ylabel("Mean trips / no-football baseline")
ax.set_title("B3.2 — Football window uplift")
plt.tight_layout()
plt.show()

best_win = b32["mean"].idxmax()
findings["B3.2"] = (
    f"Football match-zone uplift — pre_2h ×{b32.loc['pre_2h','mean']:.2f}, "
    f"during ×{b32.loc['during','mean']:.2f}, post_2h ×{b32.loc['post_2h','mean']:.2f} "
    f"(vs same zone·dow·hour without football). Largest window: {best_win}."
)
print(findings["B3.2"])


Football match-zone uplift — pre_2h ×2.17, during ×1.44, post_2h ×2.85 (vs same zone·dow·hour without football). Largest window: post_2h.


### B3.3 Event type ranking


In [20]:
no_event_base = (
    master.loc[master["in_event_window"] == 0]
    .groupby(["zone", "dow", "hour"])["trips"]
    .mean()
    .rename("base_mean")
)

etype_rows = []
for etype, grp in confirmed.dropna(subset=["zone"]).groupby("event_type"):
    ratios = []
    for _, ev in grp.iterrows():
        sub = master[
            (master["zone"] == ev["zone"])
            & (master["pickup_hour_eat"] >= ev["window_start"])
            & (master["pickup_hour_eat"] <= ev["window_end"])
        ]
        if sub.empty:
            continue
        tmp = sub.merge(no_event_base, on=["zone", "dow", "hour"], how="left")
        tmp = tmp[tmp["base_mean"] > 0]
        ratios.extend((tmp["trips"] / tmp["base_mean"]).tolist())
    if ratios:
        etype_rows.append(
            {
                "event_type": etype,
                "n_event_zone_rows": len(grp.drop_duplicates("event_id")),
                "n_matched_hours": len(ratios),
                "mean_ratio": float(np.mean(ratios)),
                "median_ratio": float(np.median(ratios)),
            }
        )
b33 = pd.DataFrame(etype_rows).sort_values("mean_ratio", ascending=False)
b33.round(3)


,event_type,n_event_zone_rows,n_matched_hours,mean_ratio,median_ratio
3,football_match,48,264,2.429,2.053
7,sports_run,3,16,1.722,0.912
0,concert,26,169,1.596,1.460
1,conference,37,267,1.342,1.286
2,exhibition,7,307,1.338,1.250
6,school_break,2,17847,1.050,0.986
4,public_holiday,13,3436,1.025,0.873
5,road_closure,23,377,0.736,0.667


In [21]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(b33["event_type"], b33["mean_ratio"], color="#00838f")
ax.axvline(1.0, color="gray", ls="--")
ax.set_xlabel("Mean demand ratio in event window")
ax.set_title("B3.3 — Event type effect size")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

strong = b33.head(3)
weak = b33[b33["mean_ratio"] < 1.05]
findings["B3.3"] = (
    "Largest effects: "
    + ", ".join(f"{r.event_type} ×{r.mean_ratio:.2f}" for r in strong.itertuples())
    + ". Weak / no clear lift: "
    + (", ".join(f"{r.event_type} ×{r.mean_ratio:.2f}" for r in weak.itertuples()) if len(weak) else "none below 1.05")
    + ". Public holidays show ratio < 1 (demand drop), which is a real effect — just opposite sign."
)
print(findings["B3.3"])


Largest effects: football_match ×2.43, sports_run ×1.72, concert ×1.60. Weak / no clear lift: public_holiday ×1.03, road_closure ×0.74. Public holidays show ratio < 1 (demand drop), which is a real effect — just opposite sign.


### B3.4 Cancelled and unlisted events


In [22]:
# Use nominal duration windows — some cancelled rows have multi-month end timestamps
cancelled = events[events["status_clean"].str.contains("cancel", na=False)].dropna(subset=["zone"])
canc_ratios = []
canc_by_event = []
for eid, grp in cancelled.groupby("event_id"):
    ev = grp.iloc[0]
    etype = ev["event_type"]
    dur_h = float(DEFAULT_DURATION_HOURS.get(etype, 2))
    pre, post = EVENT_WINDOW.get(etype, (1, 1))
    start = pd.Timestamp(ev["start_eat"])
    t0 = start - pd.Timedelta(hours=pre)
    t1 = start + pd.Timedelta(hours=dur_h + post)
    ratios = []
    for z in grp["zone"].dropna().unique():
        sub = master[
            (master["zone"] == z)
            & (master["pickup_hour_eat"] >= t0)
            & (master["pickup_hour_eat"] < t1)
        ]
        if sub.empty:
            continue
        tmp = sub.merge(no_event_base, on=["zone", "dow", "hour"], how="left")
        tmp = tmp[tmp["base_mean"] > 0]
        ratios.extend((tmp["trips"] / tmp["base_mean"]).tolist())
    if ratios:
        canc_ratios.extend(ratios)
        canc_by_event.append({"event_id": eid, "event_type": etype, "mean_ratio": float(np.mean(ratios)), "n": len(ratios)})

canc_mean = float(np.mean(canc_ratios)) if canc_ratios else np.nan
canc_event_df = pd.DataFrame(canc_by_event)
print("Cancelled events (unique ids):", cancelled["event_id"].nunique())
print("Matched zone-hours (nominal windows):", len(canc_ratios), "mean ratio:", round(canc_mean, 3) if canc_ratios else None)
display_cols = ["event_id", "event_name", "event_type", "zone", "start_eat", "status_clean"]
cancelled.drop_duplicates("event_id")[display_cols]


Cancelled events (unique ids): 6
Matched zone-hours (nominal windows): 26 mean ratio: 1.413


,event_id,event_name,event_type,zone,start_eat,status_clean
53,EVT-0113,Live concert,concert,Bole,2025-09-01 20:00:00+03:00,cancelled
76,EVT-0052,Premier league match,football_match,Kazanchis,2025-04-23 18:00:00+03:00,cancelled
191,EVT-0058,Premier league match,football_match,Kazanchis,2025-05-03 15:00:00+03:00,cancelled
226,EVT-0042,Conference: Health congress,conference,Kazanchis,2025-04-03 08:00:00+03:00,cancelled
229,EVT-0099,Live concert,concert,Bole,2025-07-28 20:00:00+03:00,cancelled
280,EVT-0020,Premier league match,football_match,Kazanchis,2025-02-22 15:00:00+03:00,cancelled


In [23]:
# Unexplained spikes: high daily zone totals vs same-dow median, no listed confirmed event that day
daily_z = master.groupby(["zone", "day"], as_index=False)["trips"].sum()
daily_z["dow"] = daily_z["day"].dt.dayofweek
med = daily_z.groupby(["zone", "dow"])["trips"].median().rename("dow_median")
daily_z = daily_z.merge(med, on=["zone", "dow"])
daily_z["ratio"] = daily_z["trips"] / daily_z["dow_median"]

ev_keys = set(
    zip(
        master.loc[master["in_event_window"] == 1, "zone"],
        master.loc[master["in_event_window"] == 1, "day"],
    )
)
daily_z["has_listed_event"] = [(z, d) in ev_keys for z, d in zip(daily_z["zone"], daily_z["day"])]
spikes = (
    daily_z[(~daily_z["has_listed_event"]) & (daily_z["ratio"] >= 1.5)]
    .sort_values("ratio", ascending=False)
    .head(12)
)
spikes[["zone", "day", "trips", "dow_median", "ratio"]]


,zone,day,trips,dow_median,ratio
431,Ayat,2025-07-20 00:00:00+03:00,1077.5,478.00,2.254184
2611,Megenagna,2025-09-10 00:00:00+03:00,1723.0,960.50,1.793857
2274,Lideta,2025-08-08 00:00:00+03:00,1266.0,731.25,1.731282
418,Ayat,2025-07-07 00:00:00+03:00,695.0,434.00,1.601382
1378,Gerji,2025-08-24 00:00:00+03:00,946.0,593.00,1.595278
2915,Merkato,2025-09-10 00:00:00+03:00,1605.0,1019.50,1.574301
2575,Megenagna,2025-08-05 00:00:00+03:00,1518.0,980.00,1.548980
1354,Gerji,2025-07-31 00:00:00+03:00,859.5,556.00,1.545863
1057,CMC,2025-08-07 00:00:00+03:00,947.0,616.50,1.536091


In [24]:
top3 = spikes.head(3)
guesses = []
for r in top3.itertuples():
    if r.zone in {"Bole"}:
        guess = "possible unlisted concert / nightlife peak"
    elif r.zone == "Megenagna":
        guess = "possible transport disruption or unlisted stadium/hub surge"
    elif r.zone == "Merkato":
        guess = "possible market festival / bulk-trading day not in calendar"
    elif r.zone == "Lideta":
        guess = "possible courthouse/office surge or unlisted conference"
    elif r.zone == "Ayat":
        guess = "possible local launch promo / estate move-in day"
    else:
        guess = "possible local gathering or church/school event not listed"
    guesses.append(f"{r.zone} {pd.Timestamp(r.day).date()} (×{r.ratio:.2f}: {guess})")

if canc_mean < 1.08:
    canc_txt = (
        f"(a) Cancelled events show little demand footprint (mean ratio {canc_mean:.2f} on "
        f"{len(canc_ratios)} nominal-window hours) — safe to exclude from features. "
    )
else:
    canc_txt = (
        f"(a) Cancelled events still show elevated demand in their nominal windows "
        f"(mean ratio {canc_mean:.2f} on {len(canc_ratios)} hours) — some 'cancelled' rows may be "
        f"mis-labeled or the crowd still came; we keep them out of features per status rules, "
        f"but treat the label cautiously. "
    )

findings["B3.4"] = canc_txt + "(b) Unlisted spikes (≥3): " + "; ".join(guesses) + "."
print(findings["B3.4"])
if len(canc_event_df):
    print(canc_event_df.round(3).to_string(index=False))


(a) Cancelled events still show elevated demand in their nominal windows (mean ratio 1.41 on 26 hours) — some 'cancelled' rows may be mis-labeled or the crowd still came; we keep them out of features per status rules, but treat the label cautiously. (b) Unlisted spikes (≥3): Ayat 2025-07-20 (×2.25: possible local launch promo / estate move-in day); Megenagna 2025-09-10 (×1.79: possible transport disruption or unlisted stadium/hub surge); Lideta 2025-08-08 (×1.73: possible courthouse/office surge or unlisted conference).
event_id     event_type  mean_ratio  n
EVT-0020 football_match       2.025  6
EVT-0042     conference       1.148  8
EVT-0052 football_match       1.402  6
EVT-0058 football_match       1.165  6


## B4 — Operations & data quality


### B4.1 Operational variables vs demand


In [25]:
ops = master[["trips", "active_drivers", "avg_wait_min", "avg_fare_birr"]].dropna()
corr = ops.corr()["trips"].drop("trips")
corr.to_frame("corr_with_trips")


,corr_with_trips
active_drivers,0.909889
avg_wait_min,0.492807
avg_fare_birr,0.004162


In [26]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col in zip(axes, ["active_drivers", "avg_wait_min", "avg_fare_birr"]):
    sample = ops.sample(min(8000, len(ops)), random_state=42)
    ax.scatter(sample[col], sample["trips"], s=4, alpha=0.25, color="#37474f")
    ax.set_xlabel(col)
    ax.set_ylabel("trips")
    ax.set_title(f"r={corr[col]:.2f}")
plt.suptitle("B4.1 — Ops variables vs trips", y=1.02)
plt.tight_layout()
plt.show()

findings["B4.1"] = (
    f"Correlations with trips: active_drivers={corr['active_drivers']:.2f}, "
    f"avg_wait_min={corr['avg_wait_min']:.2f}, avg_fare_birr={corr['avg_fare_birr']:.2f}. "
    f"Drivers co-move with demand (supply dispatched to busy zones) — not a clean cause. "
    f"Wait time can rise when demand outstrips supply (positive) or fall when oversupplied. "
    f"Fare mixes trip length/mix, not pure demand. None are known at forecast time for future hours, "
    f"so they must not be model inputs (leakage / unavailable)."
)
print(findings["B4.1"])


Correlations with trips: active_drivers=0.91, avg_wait_min=0.49, avg_fare_birr=0.00. Drivers co-move with demand (supply dispatched to busy zones) — not a clean cause. Wait time can rise when demand outstrips supply (positive) or fall when oversupplied. Fare mixes trip length/mix, not pure demand. None are known at forecast time for future hours, so they must not be model inputs (leakage / unavailable).


### B4.2 Gaps and outages


In [27]:
hours = pd.date_range(master["pickup_hour_eat"].min(), master["pickup_hour_eat"].max(), freq="h")
zone_hours = {z: set(g["pickup_hour_eat"]) for z, g in master.groupby("zone")}
non_ayat = [z for z in zone_hours if z != "Ayat"]

miss_count = {}
for h in hours:
    c = sum(1 for z in non_ayat if h not in zone_hours[z])
    if c:
        miss_count[h] = c

# Platform outage: missing in ≥10 of 11 early zones
common = sorted(h for h, c in miss_count.items() if c >= 10)

def compress(ts_list):
    if not ts_list:
        return []
    ranges = []
    start = prev = ts_list[0]
    for h in ts_list[1:]:
        if h - prev == pd.Timedelta(hours=1):
            prev = h
        else:
            ranges.append((start, prev, int((prev - start).total_seconds() // 3600) + 1))
            start = prev = h
    ranges.append((start, prev, int((prev - start).total_seconds() // 3600) + 1))
    return ranges

outage_ranges = compress(common)
ayat_start = master.loc[master["zone"] == "Ayat", "pickup_hour_eat"].min()
ayat_pre = [h for h in hours if h < ayat_start]

# Per-zone leftover gaps (not in common outage, not Ayat pre-launch)
common_set = set(common)
random_gaps = []
for z, present in zone_hours.items():
    for h in hours:
        if h in present:
            continue
        if z == "Ayat" and h < ayat_start:
            continue
        if h in common_set:
            continue
        random_gaps.append((z, h))

gap_summary = pd.DataFrame(
    [
        {"gap_class": "platform_outage_shared", "n_hours": len(common), "treatment": "left missing / model uses lags; not imputed as zeros"},
        {"gap_class": "ayat_pre_launch", "n_hours": len(ayat_pre), "treatment": "Ayat excluded before launch; not an outage"},
        {"gap_class": "sporadic_zone_gaps", "n_hours": len(random_gaps), "treatment": "remain missing zone-hours; features use available history"},
    ]
)
print("Ayat launch:", ayat_start)
print("Shared outage ranges:")
for a, b, n in outage_ranges:
    print(f"  {a} → {b} ({n}h)")
gap_summary


Ayat launch: 2025-03-15 00:00:00+03:00
Shared outage ranges:
  2025-05-12 02:00:00+03:00 → 2025-05-13 19:00:00+03:00 (42h)


,gap_class,n_hours,treatment
0,platform_outage_shared,42,left missing / model uses lags; not imputed as...
1,ayat_pre_launch,1752,Ayat excluded before launch; not an outage
2,sporadic_zone_gaps,682,remain missing zone-hours; features use availa...


In [28]:
findings["B4.2"] = (
    f"Shared platform outages: {len(common)} hours across ≥10 early zones "
    f"({len(outage_ranges)} contiguous block(s)). "
    f"Ayat late launch from {ayat_start} explains {len(ayat_pre)} missing Ayat hours before go-live. "
    f"Remaining ~{len(random_gaps)} sporadic zone-hour gaps look like random missing records. "
    f"Treatment: do not invent zero-demand for outages/pre-launch; keep master at observed zone-hours; "
    f"forecast features rely on lags/calendar rather than filling gaps with zeros."
)
print(findings["B4.2"])


Shared platform outages: 42 hours across ≥10 early zones (1 contiguous block(s)). Ayat late launch from 2025-03-15 00:00:00+03:00 explains 1752 missing Ayat hours before go-live. Remaining ~682 sporadic zone-hour gaps look like random missing records. Treatment: do not invent zero-demand for outages/pre-launch; keep master at observed zone-hours; forecast features rely on lags/calendar rather than filling gaps with zeros.


### B4.3 Pay-period effect


In [29]:
daily = (
    master.groupby("day", as_index=False)
    .agg(trips=("trips", "sum"), payday=("is_payday_window", "max"))
)
daily["dow"] = daily["day"].dt.dayofweek
daily["t"] = np.arange(len(daily))
coef = np.polyfit(daily["t"], daily["trips"], 1)
daily["detrended"] = daily["trips"] - np.polyval(coef, daily["t"])
daily["dow_mean"] = daily.groupby("dow")["detrended"].transform("mean")
daily["resid"] = daily["detrended"] - daily["dow_mean"]

pay = daily.loc[daily["payday"] == 1, "resid"]
oth = daily.loc[daily["payday"] == 0, "resid"]
lift = pay.mean() - oth.mean()
lift_pct = 100 * lift / daily["trips"].mean()
raw_ratio = daily.loc[daily["payday"] == 1, "trips"].mean() / daily.loc[daily["payday"] == 0, "trips"].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(daily.loc[daily["payday"] == 0, "day"], daily.loc[daily["payday"] == 0, "resid"], s=12, alpha=0.5, label="ordinary")
ax.scatter(daily.loc[daily["payday"] == 1, "day"], daily.loc[daily["payday"] == 1, "resid"], s=16, alpha=0.8, label="payday window")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.legend()
ax.set_title("B4.3 — Detrended+deseasoned daily residual (payday vs not)")
ax.set_ylabel("Residual trips")
plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        "metric": ["raw_mean_ratio_payday_over_other", "detrended_deseasoned_lift_trips", "lift_pct_of_mean_day"],
        "value": [raw_ratio, lift, lift_pct],
    }
)


,metric,value
0,raw_mean_ratio_payday_over_other,1.065823
1,detrended_deseasoned_lift_trips,411.997934
2,lift_pct_of_mean_day,5.149052


In [30]:
keep = abs(lift_pct) >= 2.0
findings["B4.3"] = (
    f"Payday window (day-of-month ≤3 or ≥28): raw mean daily trips ratio {raw_ratio:.3f} vs other days; "
    f"after linear detrend + day-of-week removal, residual lift ≈ {lift:,.0f} trips/day ({lift_pct:.1f}% of mean day). "
    + (
        "Effect is large enough to keep `is_payday_window` as a calendar feature."
        if keep
        else "Effect is small after detrend — optional feature, low priority vs hour/dow/rain/events."
    )
)
print(findings["B4.3"])


Payday window (day-of-month ≤3 or ≥28): raw mean daily trips ratio 1.066 vs other days; after linear detrend + day-of-week removal, residual lift ≈ 412 trips/day (5.1% of mean day). Effect is large enough to keep `is_payday_window` as a calendar feature.


## Export report


In [31]:
lines = [
    "# B — Data Analysis Report",
    "",
    "Source: `notebooks/02_analysis_report.ipynb` on `data/processed/master_train.csv`.",
    "Each task: result + short interpretation (Deliverable B, 14 pts).",
    "",
]
for key in [
    "B1.1", "B1.2", "B1.3", "B1.4",
    "B2.1", "B2.2", "B2.3",
    "B3.1", "B3.2", "B3.3", "B3.4",
    "B4.1", "B4.2", "B4.3",
]:
    section = {
        "B1.1": "### B1.1 Volume by zone",
        "B1.2": "### B1.2 Hour-of-day profile by zone type",
        "B1.3": "### B1.3 Weekday vs weekend",
        "B1.4": "### B1.4 Trend",
        "B2.1": "### B2.1 Timezone check",
        "B2.2": "### B2.2 Rain effect by zone type",
        "B2.3": "### B2.3 Rain dose-response",
        "B3.1": "### B3.1 Public holidays",
        "B3.2": "### B3.2 Football event windows",
        "B3.3": "### B3.3 Event type ranking",
        "B3.4": "### B3.4 Cancelled and unlisted events",
        "B4.1": "### B4.1 Operational variables vs demand",
        "B4.2": "### B4.2 Gaps and outages",
        "B4.3": "### B4.3 Pay-period effect",
    }[key]
    if key.startswith("B1") and key.endswith(".1"):
        lines += ["## B1 — Demand patterns", ""]
    if key == "B2.1":
        lines += ["## B2 — Weather", ""]
    if key == "B3.1":
        lines += ["## B3 — Events & calendar", ""]
    if key == "B4.1":
        lines += ["## B4 — Operations & data quality", ""]
    lines += [section, "", findings[key], ""]

out = REPORTS / "B_analysis_report.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", out)
print("Tasks complete:", len(findings), "/ 14")
assert len(findings) == 14


Wrote ../reports/B_analysis_report.md
Tasks complete: 14 / 14
